# **PROSES-ETL Dataset**

Proses ETL merupakan tahapan utama dalam pembangunan Data Warehouse yang bertujuan untuk menyiapkan data agar siap digunakan dalam analisis. Pada proyek ini, proses ETL dilakukan menggunakan Google Colab dengan bantuan bahasa pemrograman Python dan library seperti pandas.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

## **1. Extract - and Data Understanding**

Tahap extract dilakukan dengan mengambil dataset dari sumber data, yaitu file CSV yang diperoleh dari Kaggle (Video Game Sales & Gaming Evolution: Exploring videogames (1980-2023)
).

Data kemudian dibaca dan dimuat ke dalam environment Google Colab menggunakan library pandas untuk memudahkan proses pengolahan selanjutnya.

In [ ]:
vgsales = pd.read_csv('vgsales.csv')
games = pd.read_csv('games.csv')

### **Lihat Data Awal**

In [ ]:
vgsales.head()
games.head()


,Unnamed: 0,Title,Release Date,Team,Rating,Times Listed,Number of Reviews,Genres,Summary,Reviews,Plays,Playing,Backlogs,Wishlist
0,0,Elden Ring,"Feb 25, 2022","['Bandai Namco Entertainment', 'FromSoftware']",4.5,3.9K,3.9K,"['Adventure', 'RPG']","Elden Ring is a fantasy, action and open world...","[""The first playthrough of elden ring is one o...",17K,3.8K,4.6K,4.8K
1,1,Hades,"Dec 10, 2019",['Supergiant Games'],4.3,2.9K,2.9K,"['Adventure', 'Brawler', 'Indie', 'RPG']",A rogue-lite hack and slash dungeon crawler in...,['convinced this is a roguelike for people who...,21K,3.2K,6.3K,3.6K
2,2,The Legend of Zelda: Breath of the Wild,"Mar 03, 2017","['Nintendo', 'Nintendo EPD Production Group No...",4.4,4.3K,4.3K,"['Adventure', 'RPG']",The Legend of Zelda: Breath of the Wild is the...,['This game is the game (that is not CS:GO) th...,30K,2.5K,5K,2.6K
3,3,Undertale,"Sep 15, 2015","['tobyfox', '8-4']",4.2,3.5K,3.5K,"['Adventure', 'Indie', 'RPG', 'Turn Based Stra...","A small child falls into the Underground, wher...",['soundtrack is tied for #1 with nier automata...,28K,679,4.9K,1.8K
4,4,Hollow Knight,"Feb 24, 2017",['Team Cherry'],4.4,3K,3K,"['Adventure', 'Indie', 'Platform']",A 2D metroidvania with an emphasis on close co...,"[""this games worldbuilding is incredible, with...",21K,2.4K,8.3K,2.3K


### **Cek Struktur Data**

In [ ]:
vgsales.info()
print("")
games.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16598 entries, 0 to 16597
Data columns (total 11 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Rank          16598 non-null  int64  
 1   Name          16598 non-null  object 
 2   Platform      16598 non-null  object 
 3   Year          16327 non-null  float64
 4   Genre         16598 non-null  object 
 5   Publisher     16540 non-null  object 
 6   NA_Sales      16598 non-null  float64
 7   EU_Sales      16598 non-null  float64
 8   JP_Sales      16598 non-null  float64
 9   Other_Sales   16598 non-null  float64
 10  Global_Sales  16598 non-null  float64
dtypes: float64(6), int64(1), object(4)
memory usage: 1.4+ MB

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1512 entries, 0 to 1511
Data columns (total 14 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Unnamed: 0         1512 non-null   int64  
 1   Title        

### **Cek Missing Value**

In [ ]:
vgsales.isnull().sum()

,0
Rank,0
Name,0
Platform,0
Year,271
Genre,0
Publisher,58
NA_Sales,0
EU_Sales,0
JP_Sales,0
Other_Sales,0


In [ ]:
games.isnull().sum()

,0
Unnamed: 0,0
Title,0
Release Date,0
Team,1
Rating,13
Times Listed,0
Number of Reviews,0
Genres,0
Summary,1
Reviews,0


### **Cek Data Duplikat**

In [ ]:
vgsales.duplicated().sum()

np.int64(0)

In [ ]:
games.duplicated().sum()

np.int64(0)

### **Idenfikasi Join Key**

In [ ]:
vgsales[['Name']].head()

,Name
0,Wii Sports
1,Super Mario Bros.
2,Mario Kart Wii
3,Wii Sports Resort
4,Pokemon Red/Pokemon Blue


In [ ]:
games[['Title']].head()

,Title
0,Elden Ring
1,Hades
2,The Legend of Zelda: Breath of the Wild
3,Undertale
4,Hollow Knight


Berdasarkan hasil eksplorasi awal, ditemukan bahwa kedua dataset memiliki struktur yang berbeda, terutama pada penamaan kolom dan format data. Dataset vgsales berfokus pada data penjualan, sedangkan dataset games berisi informasi tambahan seperti rating dan ulasan. Selain itu, terdapat kemungkinan perbedaan format penulisan nama game yang akan digunakan sebagai penghubung antar dataset, sehingga diperlukan proses pembersihan data sebelum dilakukan penggabungan.

## **2. Transform - Cleaning Data**

Tahap transform melibatkan pembersihan, transformasi, dan pengayaan data agar sesuai dengan kebutuhan analisis.

### **Cleaning vgsales dataset**

In [ ]:
vgsales_clean = vgsales.copy()

# Hapus data tanpa tahun
vgsales_clean = vgsales_clean.dropna(subset=['Year'])

# Ubah Year ke integer
vgsales_clean['Year'] = vgsales_clean['Year'].astype(int)

# Isi publisher kosong dengan 'Unknown'
vgsales_clean['Publisher'] = vgsales_clean['Publisher'].fillna('Unknown')

### **Cleaning games dataset**

In [ ]:
games_clean = games.copy()

# Drop kolom gak penting
games_clean = games_clean.drop(columns=['Unnamed: 0'])

# Isi rating kosong dengan rata-rata
games_clean['Rating'] = games_clean['Rating'].fillna(games_clean['Rating'].mean())

### **Cleaning Kolom Angka**

In [ ]:
def convert_to_number(x):
    if isinstance(x, str):
        x = x.replace(',', '')
        if 'K' in x:
            return float(x.replace('K', '')) * 1_000
        elif 'M' in x:
            return float(x.replace('M', '')) * 1_000_000
    return float(x)

cols = ['Number of Reviews', 'Plays', 'Playing', 'Backlogs', 'Wishlist']

for col in cols:
    games_clean[col] = games_clean[col].apply(convert_to_number)

### **Normaliasi Nama Biar di Join**

In [ ]:
import re

def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-z0-9 ]', '', text)
    return text.strip()

vgsales_clean['Name_clean'] = vgsales_clean['Name'].apply(clean_text)
games_clean['Title_clean'] = games_clean['Title'].apply(clean_text)

### **JOIN DATASET**

In [ ]:
df_merged = pd.merge(
    vgsales_clean,
    games_clean,
    left_on='Name_clean',
    right_on='Title_clean',
    how='left'
)

Cek hasil join

In [ ]:
df_merged[['Name', 'Title', 'Rating']].head()

,Name,Title,Rating
0,Wii Sports,Wii Sports,3.7
1,Wii Sports,Wii Sports,3.7
2,Wii Sports,Wii Sports,3.7
3,Super Mario Bros.,Super Mario Bros.,3.5
4,Mario Kart Wii,Mario Kart Wii,3.9


Pada tahap transformasi, dilakukan pembersihan data untuk mengatasi nilai kosong, memperbaiki tipe data, serta mengubah format data numerik yang sebelumnya berbentuk string menjadi numerik. Selain itu, dilakukan normalisasi pada nama game untuk memastikan konsistensi dalam proses penggabungan data.

### **Multi-Genre**

Membuat sebuah colom agar bisa memgabungkan kolom genre dari dataset games.csv dengan kolom genre vgsales yang di mana 1 game bisa memilik banyak genre.

In [ ]:
games_genre = games_clean[['Title_clean', 'Genres']].copy()
games_genre['Genres'] = games_genre['Genres'].str.split(',')
games_genre = games_genre.explode('Genres')
games_genre['Genres'] = games_genre['Genres'].str.strip()

Merge. Menggabungkannya ke data

In [ ]:
df_multi = df_merged.merge(
    games_genre,
    left_on='Name_clean',
    right_on='Title_clean',
    how='left'
)

## **3. Membentuk Data Warehouse (Fact & Dimension Tables)**

### **DIMENSION Tables**

1. dim_game

In [ ]:
dim_game = df_merged[['Name', 'Rating', 'Number of Reviews', 'Plays']].copy()

dim_game = dim_game.drop_duplicates()

# Generate ID
dim_game['game_id'] = range(1, len(dim_game) + 1)

# Reorder kolom
dim_game = dim_game[['game_id', 'Name', 'Rating', 'Number of Reviews', 'Plays']]

2. dim_platform

In [ ]:
dim_platform = df_merged[['Platform']].drop_duplicates()

dim_platform['platform_id'] = range(1, len(dim_platform) + 1)

dim_platform = dim_platform[['platform_id', 'Platform']]

3. dim_publisher

In [ ]:
dim_publisher = df_merged[['Publisher']].drop_duplicates()

dim_publisher['publisher_id'] = range(1, len(dim_publisher) + 1)

dim_publisher = dim_publisher[['publisher_id', 'Publisher']]

4. dim_time

In [ ]:
dim_time = df_merged[['Year']].drop_duplicates()

dim_time['time_id'] = range(1, len(dim_time) + 1)

dim_time = dim_time[['time_id', 'Year']]

5. dim_genre

In [ ]:
dim_genre = df_multi[['Genres_y']].drop_duplicates().reset_index(drop=True)

dim_genre = dim_genre.rename(columns={'Genres_y': 'Genres'})

dim_genre['genre_id'] = range(1, len(dim_genre) + 1)

dim_genre = dim_genre[['genre_id', 'Genres']]

6. bridge_game_genre

In [ ]:
bridge = df_multi[['Name', 'Genres_y']].drop_duplicates()

bridge = bridge.rename(columns={'Genres_y': 'Genres'})

# join ke dim_game
bridge = bridge.merge(dim_game[['game_id', 'Name']], on='Name', how='left')

# join ke dim_genre
bridge = bridge.merge(dim_genre, on='Genres', how='left')

bridge_game_genre = bridge[['game_id', 'genre_id']].drop_duplicates()

### **MAPPING (Merge ID ke data)**

In [ ]:
df_dw = df_merged.copy()

# game_id
df_dw = df_dw.merge(dim_game[['game_id', 'Name']], on='Name', how='left')

# platform
df_dw = df_dw.merge(dim_platform, on='Platform', how='left')

# publisher
df_dw = df_dw.merge(dim_publisher, on='Publisher', how='left')

# time
df_dw = df_dw.merge(dim_time, on='Year', how='left')

### **FACT Table**

In [ ]:
fact_sales = df_dw[[
    'game_id',
    'platform_id',
    'publisher_id',
    'time_id',
    'NA_Sales',
    'EU_Sales',
    'JP_Sales',
    'Other_Sales',
    'Global_Sales'
]]

Validasi Id cek

In [ ]:
fact_sales.isnull().sum()

,0
game_id,0
platform_id,0
publisher_id,0
time_id,0
NA_Sales,0
EU_Sales,0
JP_Sales,0
Other_Sales,0
Global_Sales,0


In [ ]:
bridge_game_genre.isnull().sum()

,0
game_id,0
genre_id,0


Pada tahap ini, data yang telah melalui proses ETL dipisahkan ke dalam tabel fakta dan tabel dimensi sesuai dengan desain Star Schema. Setiap tabel dimensi memiliki primary key yang digunakan sebagai penghubung ke tabel fakta. Proses ini bertujuan untuk meningkatkan efisiensi query dan mempermudah analisis data dalam lingkungan Data Warehouse.

## **4. LOAD ke Database**

In [ ]:
import pandas as pd

# List semua dataframe
tables = {
    "dim_game": dim_game,
    "dim_platform": dim_platform,
    "dim_publisher": dim_publisher,
    "dim_time": dim_time,
    "dim_genre": dim_genre,
    "bridge_game_genre": bridge_game_genre,
    "fact_sales": fact_sales
}

# Membuat file SQL
with open("game_dw_mysql.sql", "w", encoding="utf-8") as f:

    for table_name, df in tables.items():

        # DROP TABLE
        f.write(f"DROP TABLE IF EXISTS `{table_name}`;\n")

        # CREATE TABLE
        columns = []

        for col, dtype in zip(df.columns, df.dtypes):

            if "int" in str(dtype):
                sql_type = "INT"
            elif "float" in str(dtype):
                sql_type = "FLOAT"
            else:
                sql_type = "TEXT"

            columns.append(f"`{col}` {sql_type}")

        create_table_sql = f"""
CREATE TABLE `{table_name}` (
    {", ".join(columns)}
);
"""

        f.write(create_table_sql)

        # INSERT INTO
        for _, row in df.iterrows():

            values = []

            for val in row:

                if pd.isna(val):
                    values.append("NULL")

                elif isinstance(val, str):
                    val = val.replace("'", "''")
                    values.append(f"'{val}'")

                else:
                    values.append(str(val))

            insert_sql = f"""
INSERT INTO `{table_name}` VALUES ({", ".join(values)});
"""

            f.write(insert_sql)

        f.write("\n\n")

print("File game_dw_mysql.sql berhasil dibuat!")

File game_dw_mysql.sql berhasil dibuat!


Validasi/Cek-cek

In [ ]:
query = "SELECT name FROM sqlite_master WHERE type='table';"
tables = pd.read_sql(query, conn)
tables

,name
0,dim_game
1,dim_platform
2,dim_publisher
3,dim_time
4,dim_genre
5,bridge_game_genre
6,fact_sales


## Pengecekan Missing Values

In [ ]:
fact_sales.isnull().sum()

,0
game_id,0
platform_id,0
publisher_id,0
time_id,0
NA_Sales,0
EU_Sales,0
JP_Sales,0
Other_Sales,0
Global_Sales,0


In [ ]:
dim_game.isnull().sum()

,0
game_id,0
Name,0
Rating,10880
Number of Reviews,10880
Plays,10880


In [ ]:
dim_platform.isnull().sum()

,0
platform_id,0
Platform,0


In [ ]:
dim_publisher.isnull().sum()

,0
publisher_id,0
Publisher,0


## Pengecekan Duplicated Values

In [ ]:
dim_platform.duplicated().sum()

np.int64(0)

In [ ]:
fact_sales[fact_sales.duplicated()]

,game_id,platform_id,publisher_id,time_id,NA_Sales,EU_Sales,JP_Sales,Other_Sales,Global_Sales
1,1,1,1,1,41.49,29.02,3.77,8.46,82.74
2,1,1,1,1,41.49,29.02,3.77,8.46,82.74
5,3,1,1,3,15.85,12.88,3.79,3.31,35.82
6,3,1,1,3,15.85,12.88,3.79,3.31,35.82
11,6,3,1,6,23.20,2.26,4.22,0.58,30.26
...,...,...,...,...,...,...,...,...,...
15485,4890,15,12,32,0.01,0.01,0.00,0.00,0.02
16153,10820,10,13,22,0.00,0.00,0.02,0.00,0.02
16373,306,5,2,3,0.00,0.01,0.00,0.00,0.01
16481,11070,15,22,30,0.00,0.01,0.00,0.00,0.01


Buat tutup koneksi ke database

In [ ]:
conn.close()

Data Warehouse dalam bentuk database.

## **5. Query Analisis**

### **Top 10 Game Terlaris**

In [ ]:
import pandas as pd
import sqlite3

conn = sqlite3.connect('game_dw.db')

query_top_game = """
SELECT g.Name, SUM(f.Global_Sales) AS total_sales
FROM fact_sales f
JOIN dim_game g ON f.game_id = g.game_id
GROUP BY g.Name
ORDER BY total_sales DESC
LIMIT 10;
"""

df_top_game = pd.read_sql(query_top_game, conn)
df_top_game

,Name,total_sales
0,Wii Sports,248.22
1,Tetris,215.04
2,Minecraft,189.84
3,Grand Theft Auto V,167.76
4,Mario Kart Wii,107.46
5,Final Fantasy VII,77.76
6,Super Mario Bros. 3,67.44
7,Super Mario 64,66.93
8,Resident Evil 2,64.40
9,New Super Mario Bros.,60.02


### **Penjualan Berdasarkan Platform**

In [ ]:
query_platform = """
SELECT p.Platform, SUM(f.Global_Sales) AS total_sales
FROM fact_sales f
JOIN dim_platform p ON f.platform_id = p.platform_id
GROUP BY p.Platform
ORDER BY total_sales DESC;
"""

df_platform = pd.read_sql(query_platform, conn)
df_platform

,Platform,total_sales
0,PS2,1327.75
1,Wii,1274.97
2,X360,1223.47
3,PS3,1199.62
4,PS,913.62
5,DS,885.75
6,GB,417.76
7,PS4,382.48
8,GBA,333.57
9,NES,323.30


### **Penjualan Berdasarkan Genre**

In [ ]:
query_genre = """
SELECT g.Genres, SUM(f.Global_Sales) AS total_sales
FROM fact_sales f
JOIN bridge_game_genre bg ON f.game_id = bg.game_id
JOIN dim_genre g ON bg.genre_id = g.genre_id
GROUP BY g.Genres
ORDER BY total_sales DESC;
"""
df_genre = pd.read_sql(query_genre, conn)
df_genre

,Genres,total_sales
0,None,6693.80
1,['Adventure',2358.85
2,'Shooter'],776.31
3,'Platform'],649.59
4,'RPG'],464.79
5,['Shooter'],437.06
6,'Puzzle'],308.83
7,'Sport'],258.16
8,['Simulator',250.61
9,'Simulator'],232.13


### **Rating VS Penjualan Rata-rata**

In [ ]:
query_rating = """
SELECT
    d.Rating,
    AVG(f.Global_Sales) AS avg_sales
FROM fact_sales f
JOIN dim_game d ON f.game_id = d.game_id
WHERE d.Rating IS NOT NULL
GROUP BY d.Rating
ORDER BY d.Rating DESC;
"""

df_rating = pd.read_sql(query_rating, conn)
df_rating

,Rating,avg_sales
0,4.6,0.550000
1,4.5,2.099412
2,4.4,1.302381
3,4.3,2.649197
4,4.2,3.283739
5,4.1,3.419901
6,4.0,1.992655
7,3.9,2.481587
8,3.8,4.446957
9,3.7,4.541695


## **5.1 Visualisasi**